Create embedding based on title,original_language,genre and overview. use the embedding in chatbot

In [0]:
spark.sql("USE moviebuff.default")

from pyspark.sql.functions import col, concat_ws, array_join

In [0]:
movies = spark.table("silver_dim_movies")

movies_text = movies.withColumn(
    "text_for_embedding",
    concat_ws(
        ". ",
        col("title"),
        col("language_name"),
        array_join(col("genres"), ", "),
        col("overview")
    )
)

movies_text.select("movie_id", "title","language_name", "text_for_embedding").show(5, truncate=80)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

test_response = w.serving_endpoints.query(
    name="databricks-bge-large-en",
    input=["Action Adventure movie about a hero saving the world"]
)

print(test_response.data[0].embedding[:5])  

In [0]:
import pandas as pd
import time

movies_pdf = movies_text.select("movie_id", "title","language_name", "text_for_embedding").toPandas()

embeddings_list = []

for idx, row in movies_pdf.iterrows():
    try:
        resp = w.serving_endpoints.query(
            name="databricks-bge-large-en",
            input=[row["text_for_embedding"]]
        )
        embedding = resp.data[0].embedding
        embeddings_list.append(embedding)
    except Exception as e:
        print(f"Failed for movie_id {row['movie_id']}: {e}")
        embeddings_list.append(None)
    
    if idx % 1000 == 0:
        print(f"Processed {idx}/{len(movies_pdf)}")

movies_pdf["embedding"] = embeddings_list
print(f"Done. {movies_pdf['embedding'].isna().sum()} failures out of {len(movies_pdf)}")

In [0]:
movies_pdf.count()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, FloatType

# drop any rows where embedding failed
movies_pdf_clean = movies_pdf.dropna(subset=["embedding"])

schema = StructType([
    StructField("movie_id", IntegerType()),
    StructField("title", StringType()),
    StructField("language_name", StringType()),
    StructField("text_for_embedding", StringType()),
    StructField("embedding", ArrayType(FloatType()))
])

embeddings_df = spark.createDataFrame(movies_pdf_clean, schema=schema)

embeddings_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_movie_embeddings")

print(f"Saved {embeddings_df.count()} movie embeddings to gold_movie_embeddings")

In [0]:
%sql
SELECT *
FROM gold_movie_embeddings
LIMIT 5;